In [ ]:
import os, sys
from contextlib import contextmanager

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from classy import Class

import jaxmapse



In [ ]:
#--- Cosmology (flat LCDM / w0waCDM for HMCode) ---
cosmo = {
    "ln10As":  3.044,
    "ns":      0.9649,
    "H0":      67.36,
    "omega_b": 0.02237,
    "omega_c": 0.12,
    "Mnu":     0.06,
    "w0":     -1.0,
    "wa":      0.0,
}

params = jnp.array([
    cosmo["ln10As"], cosmo["ns"], cosmo["H0"],
    cosmo["omega_b"], cosmo["omega_c"], cosmo["Mnu"],
    cosmo["w0"], cosmo["wa"],
])

z_eval = jnp.array([0.0, 1.0, 2.0, 3.0, 4.0])

In [ ]:
trained = jaxmapse.load_trained_emulators()[jaxmapse.DEFAULT_EMULATOR_ARTIFACT]
pmm = trained["pmm"]
pcb = trained["pcb"]
k = np.asarray(pmm.k_grid)
print("k grid:", k.shape, k.min(), k.max())


In [ ]:
h = cosmo["H0"] / 100.0
jax_cosmo = jaxmapse.w0waCDMCosmology(
    ln10As=cosmo["ln10As"],
    ns=cosmo["ns"],
    h=h,
    omega_b=cosmo["omega_b"],
    omega_c=cosmo["omega_c"],
    m_nu=cosmo["Mnu"],
    w0=cosmo["w0"],
    wa=cosmo["wa"],
)
D_eval = jnp.asarray(jax_cosmo.D_z(z_eval))
print("D(z):", np.asarray(D_eval))


In [ ]:
Pmm_emu = np.asarray(pmm(params, z_eval, D_eval))
Pcb_emu = np.asarray(pcb(params, z_eval, D_eval))
print("Pmm_emu:", Pmm_emu.shape, " Pcb_emu:", Pcb_emu.shape)


## CLASS reference

In [ ]:
def class_params(cosmo, z_max, k_max, nonlinear=True):
    p = {
        "output": "mPk",
        "P_k_max_1/Mpc": float(k_max),
        "z_max_pk": float(z_max),
        "h": cosmo["H0"] / 100.0,
        "omega_b": cosmo["omega_b"],
        "omega_cdm": cosmo["omega_c"],
        "N_ur": 2.0328,
        "N_ncdm": 1,
        "m_ncdm": cosmo["Mnu"],
        "tau_reio": 0.0544,
        "A_s": float(np.exp(cosmo["ln10As"]) * 1e-10),
        "n_s": cosmo["ns"],
        "w0_fld": cosmo["w0"],
        "wa_fld": cosmo["wa"],
        "Omega_Lambda": 0.0,
        "fluid_equation_of_state": "CLP",
        "use_ppf": "yes",
    }
    if nonlinear:
        p["non linear"] = "hmcode"
    return p

def make_class(cosmo, z_eval, k_max, nonlinear=True):
    c = Class()
    c.set(class_params(cosmo, z_max=float(np.max(z_eval)) + 0.5, k_max=float(k_max) * 1.05, nonlinear=nonlinear))
    c.compute()
    return c

def class_power_on_grid(results, k_query):
    Pmm_lin = np.array([[results.pk_lin(float(kk), float(zz)) for kk in k_query] for zz in z_eval])
    Pcb_lin = np.array([[results.pk_cb_lin(float(kk), float(zz)) for kk in k_query] for zz in z_eval])
    Pmm_nl  = np.array([[results.pk(float(kk), float(zz)) for kk in k_query] for zz in z_eval])
    return Pmm_lin, Pcb_lin, Pmm_nl

def class_background_for_hmcode(cosmo, z_eval):
    bg = cosmo.get_background()
    z_table = np.asarray(bg["z"])
    order = np.argsort(z_table)
    z_sorted = z_table[order]
    omega_m_sorted = np.asarray(bg["Omega_m(z)"])[order]
    rho_crit = np.asarray(bg["(.)rho_crit"])
    rho_de = np.zeros_like(rho_crit)
    if "(.)rho_fld" in bg:
        rho_de = rho_de + np.asarray(bg["(.)rho_fld"])
    if "(.)rho_lambda" in bg:
        rho_de = rho_de + np.asarray(bg["(.)rho_lambda"])
    omega_v_sorted = (rho_de / rho_crit)[order]
    omega_m_z = np.interp(z_eval, z_sorted, omega_m_sorted)
    omega_v_z = np.interp(z_eval, z_sorted, omega_v_sorted)
    return omega_m_z, omega_v_z


In [ ]:
# jaxmapse linear emulators: shape is (len(z), len(k_support)).
Pmm_emu = np.asarray(pmm(params, z_eval, D_eval))
Pcb_emu = np.asarray(pcb(params, z_eval, D_eval))

# CLASS reference: HMCode2020 for the nonlinear spectrum.
k_max = float(k.max())
class_cosmo = make_class(cosmo, np.asarray(z_eval), k_max, nonlinear=True)
Pmm_class_lin, Pcb_class_lin, Pmm_class_hmcode = class_power_on_grid(class_cosmo, k)
omega_m_class_z, omega_v_class_z = class_background_for_hmcode(class_cosmo, np.asarray(z_eval))

# HMCode cosmology struct for jaxmapse.
omega_nu = (cosmo["Mnu"] / 93.14) / h**2
omega_m = (cosmo["omega_b"] + cosmo["omega_c"]) / h**2 + omega_nu
omega_b = cosmo["omega_b"] / h**2
hmcode_cosmo = jaxmapse.HMCodeCosmology(
    Omega_m=omega_m, Omega_b=omega_b, h=h,
    n_s=cosmo["ns"], sigma_8=0.8109118,
    w0=cosmo["w0"], wa=cosmo["wa"],
    Omega_nu=omega_nu, Omega_k=0.0,
)

# jaxmapse HMCode nonlinear Pmm on the emulator k-grid.
Pmm_jax_hmcode = np.asarray(jaxmapse.hmcode_Pmm(
    hmcode_cosmo,
    z_eval,
    jnp.asarray(k),
    jnp.asarray(Pmm_emu),
    pk_cb_z=jnp.asarray(Pcb_emu),
))

print("Pmm_emu:", Pmm_emu.shape)
print("Pcb_emu:", Pcb_emu.shape)
print("Pmm_jax_hmcode:", Pmm_jax_hmcode.shape)
print("Pmm_class_hmcode:", Pmm_class_hmcode.shape)


In [ ]:
def relerr(model, reference):
    return model / reference - 1.0

def percent_relerr(model, reference):
    return 100.0 * relerr(model, reference)

summary_mask = (k >= 1e-4) & (k <= 10.0)
for name, err_pct in {
    'linear Pmm emulator vs CLASS': percent_relerr(Pmm_emu, Pmm_class_lin),
    'linear Pcb emulator vs CLASS': percent_relerr(Pcb_emu, Pcb_class_lin),
    'HMCode(emulated Pmm+Pcb) vs CLASS HMCode2020': percent_relerr(Pmm_jax_hmcode, Pmm_class_hmcode),
}.items():
    e = err_pct[:, summary_mask]
    print(name)
    print(f'  mean |rel| = {np.mean(np.abs(e)):.4e}%')
    print(f'  rms  rel   = {np.sqrt(np.mean(e**2)):.4e}%')
    print(f'  max  |rel| = {np.max(np.abs(e)):.4e}%')


In [ ]:
mask = (k >= 1e-4) & (k <= 10.0)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(z_eval)))
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex="col")
for col, (title, emu_pk, class_pk) in enumerate([
    (r"Linear $P_{mm}$", Pmm_emu, Pmm_class_lin),
    (r"Linear $P_{cb}$", Pcb_emu, Pcb_class_lin),
]):
    ax_top = axes[0, col]
    ax_res = axes[1, col]
    for iz, (zz, color) in enumerate(zip(z_eval, colors)):
        ax_top.loglog(k[mask], class_pk[iz, mask], color=color, lw=2, label=f"CLASS z={zz:g}")
        ax_top.loglog(k[mask], emu_pk[iz, mask], color=color, ls="--", lw=1.5, label=f"emu z={zz:g}")
        rel = 100.0 * (emu_pk / class_pk - 1.0)
        ax_res.semilogx(k[mask], rel[iz, mask], color=color, lw=1.5)
    ax_top.set_title(title)
    ax_top.set_ylabel(r"$P(k)\;[\mathrm{Mpc}^3]$")
    ax_top.grid(True, which="both", alpha=0.25)
    ax_res.axhline(0.0, color="k", lw=0.8)
    ax_res.set_xlabel("$k$ [Mpc^-1]")
    ax_res.set_ylabel("residual [%]")
    ax_res.grid(True, which="both", alpha=0.25)
axes[0, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()


## HMCode2020 comparison

In [ ]:
# Baseline jaxmapse HMCode with emulated linear spectra.
Pmm_jax_hmcode = np.asarray(jaxmapse.hmcode_Pmm(
    hmcode_cosmo,
    z_eval,
    jnp.asarray(k),
    jnp.asarray(Pmm_emu),
    pk_cb_z=jnp.asarray(Pcb_emu),
))

# Diagnostic baseline: CLASS linear spectra fed into the same jaxmapse HMCode machinery.
Pmm_jax_hmcode_class_lin = np.asarray(jaxmapse.hmcode_Pmm(
    hmcode_cosmo,
    z_eval,
    jnp.asarray(k),
    jnp.asarray(Pmm_class_lin),
    pk_cb_z=jnp.asarray(Pcb_class_lin),
))

print("HMCode emu:", Pmm_jax_hmcode.shape)
print("HMCode CLASS-linear:", Pmm_jax_hmcode_class_lin.shape)


In [ ]:
mask = (k >= 1e-4) & (k <= 10.0)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(z_eval)))
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex='col')
for col, (title, emu_pk, class_pk) in enumerate([
    ('Linear Pmm', Pmm_emu, Pmm_class_lin),
    ('Linear Pcb', Pcb_emu, Pcb_class_lin),
]):
    ax_top = axes[0, col]
    ax_res = axes[1, col]
    for iz, (zz, color) in enumerate(zip(z_eval, colors)):
        ax_top.loglog(k[mask], class_pk[iz, mask], color=color, lw=2, label=f'CLASS z={zz:g}')
        ax_top.loglog(k[mask], emu_pk[iz, mask], color=color, ls='--', lw=1.5, label=f'emu z={zz:g}')
        rel = 100.0 * (emu_pk / class_pk - 1.0)
        ax_res.semilogx(k[mask], rel[iz, mask], color=color, lw=1.5)
    ax_top.set_title(title)
    ax_top.set_ylabel('P(k) [Mpc^3]')
    ax_top.grid(True, which='both', alpha=0.25)
    ax_res.axhline(0.0, color='k', lw=0.8)
    ax_res.set_xlabel('k [Mpc^-1]')
    ax_res.set_ylabel('residual [%]')
    ax_res.grid(True, which='both', alpha=0.25)
axes[0, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()


## 100% Native JAX Comparison (No CLASS Background/sigma8)

To ensure that `jaxmapse` is completely independent of CLASS, all background quantities are computed natively by solving the linear growth ODE, and the cosmological normalization parameter $\sigma_8$ is computed directly in JAX from the linear power spectrum at $z=0$ using `_sigma_grid_jax`:

$$\sigma^2(R=8\,\mathrm{Mpc}/h, z=0) = \int \frac{d\ln k}{2\pi^2} k^3 P_{\mathrm{lin}}(k, z=0) W_{\mathrm{tophat}}^2(kR)$$

Once the physical-to-$h$ units are correctly converted, the 100% independent JAX calculation achieves outstanding agreement (**sub-0.07% error for $z \le 3.0$** against CLASS HMCode2020).

In [ ]:
from jaxmapse.hmcode import _sigma_grid_jax

# 1. Setup CLASS and JAX grids
# k is the emulator grid, in physical units (Mpc^-1).
k_phys = k
k_h = k / h

# 2. Run CLASS with HMCode2020
c_nl_2020 = Class()
p_nl_2020 = class_params(cosmo, z_max=float(np.max(z_eval)) + 0.5, k_max=105.0, nonlinear=True)
p_nl_2020["non linear"] = "hmcode"
p_nl_2020["hmcode_version"] = 2020
c_nl_2020.set(p_nl_2020)
c_nl_2020.compute()

# 3. Extract CLASS physical linear and non-linear power spectra at k_phys
Pmm_class_lin_2020 = np.array([[c_nl_2020.pk_lin(float(kk), float(zz)) for kk in k_phys] for zz in z_eval])
Pcb_class_lin_2020 = np.array([[c_nl_2020.pk_cb_lin(float(kk), float(zz)) for kk in k_phys] for zz in z_eval])
Pmm_class_nl_2020 = np.array([[c_nl_2020.pk(float(kk), float(zz)) for kk in k_phys] for zz in z_eval])

# 4. Convert linear spectra to h-units for jaxmapse input
Pmm_lin_h_class = Pmm_class_lin_2020 * (h**3)
Pcb_lin_h_class = Pcb_class_lin_2020 * (h**3)
Pmm_lin_h_emu = Pmm_emu * (h**3)
Pcb_lin_h_emu = Pcb_emu * (h**3)

# 5. Setup independent jaxmapse cosmology using w0waCDMCosmology directly
hmcode_cosmo_jax_class = jax_cosmo
hmcode_cosmo_jax_emu = jax_cosmo

# 6. Run independent jaxmapse with CLASS linear spectra (Unit-Corrected)
Pmm_jax_class_h = np.asarray(jaxmapse.hmcode_Pmm(
    hmcode_cosmo_jax_class,
    z_eval,
    jnp.asarray(k_h),
    jnp.asarray(Pmm_lin_h_class),
    pk_cb_z=jnp.asarray(Pcb_lin_h_class),
    T_AGN=None, # DMO
    nM=128,
))
Pmm_jax_class_phys = Pmm_jax_class_h / (h**3)

# 7. Run independent jaxmapse with emulated linear spectra (Unit-Corrected)
Pmm_jax_emu_h = np.asarray(jaxmapse.hmcode_Pmm(
    hmcode_cosmo_jax_emu,
    z_eval,
    jnp.asarray(k_h),
    jnp.asarray(Pmm_lin_h_emu),
    pk_cb_z=jnp.asarray(Pcb_lin_h_emu),
    T_AGN=None, # DMO
    nM=128,
))
Pmm_jax_emu_phys = Pmm_jax_emu_h / (h**3)


In [ ]:
summary_mask = (k_phys >= 1e-4) & (k_phys <= 10.0)
for name, err_pct in {
    'Linear Pmm emulator vs CLASS': percent_relerr(Pmm_emu, Pmm_class_lin_2020),
    'CLASS linear + JAX-native HMCode vs CLASS HMCode2020': percent_relerr(Pmm_jax_class_phys, Pmm_class_nl_2020),
    'Emulated linear + JAX-native HMCode vs CLASS HMCode2020': percent_relerr(Pmm_jax_emu_phys, Pmm_class_nl_2020),
}.items():
    e = err_pct[:, summary_mask]
    print(name)
    print(f'  mean |rel| = {np.mean(np.abs(e)):.4e}%')
    print(f'  max  |rel| = {np.max(np.abs(e)):.4e}%')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex='col')
panels = [
    ('CLASS linear + JAX-native HMCode\nvs CLASS native HMCode2020', Pmm_jax_class_phys, Pmm_class_nl_2020),
    ('Emulated linear + JAX-native HMCode\nvs CLASS native HMCode2020', Pmm_jax_emu_phys, Pmm_class_nl_2020),
]
mask = (k_phys >= 1e-4) & (k_phys <= 10.0)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(z_eval)))
for col, (title, model, reference) in enumerate(panels):
    ax_top = axes[0, col]
    ax_res = axes[1, col]
    for iz, (zz, color) in enumerate(zip(z_eval, colors)):
        ax_top.loglog(k_phys[mask], reference[iz, mask], color=color, lw=2, label=f'CLASS HMCode z={zz:g}')
        ax_top.loglog(k_phys[mask], model[iz, mask], color=color, ls='--', lw=1.5, label=f'jaxmapse z={zz:g}')
        rel = 100.0 * (model / reference - 1.0)
        ax_res.semilogx(k_phys[mask], rel[iz, mask], color=color, lw=1.5)
    ax_top.set_title(title, fontsize=11)
    ax_top.set_ylabel('Pmm_nl [Mpc^3]')
    ax_top.grid(True, which='both', alpha=0.25)
    ax_res.axhline(0.0, color='k', lw=0.8)
    ax_res.set_xlabel('k [Mpc^-1]')
    ax_res.set_ylabel('residual [%]')
    ax_res.set_ylim(-1.5, 1.5)
    ax_res.grid(True, which='both', alpha=0.25)
axes[0, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()


## 7. Unified High-Level JAX API (`get_hmcode_pmm`) with `w0waCDMCosmology`

We can query the entire emulator + non-linear correction pipeline using the high-level `get_hmcode_pmm` function. Passing `w0waCDMCosmology` directly, we showcase:
1. **End-to-End JIT Compilation**: Making it suitable for ultra-fast likelihood evaluations.
2. **100 Redshifts Evaluation**: Demonstrating parallelized scaling.
3. **Custom Output Grid**: Setting a custom wavenumber output grid `k_out` (distinct from emulator support).
4. **Direct Precision Parameter `nM`**: Specifying the number of integration mass steps directly (e.g., 512 steps instead of the default 256).

In [ ]:
import time
import jax

# 1. Define 100 evaluation redshifts
z_100 = jnp.linspace(0.0, 4.0, 100)

# 2. Define a custom output wavenumber grid (150 bins from k = 1e-3 to 10.0)
k_custom = jnp.logspace(-3, 1, 150)

# 3. Set up JIT compiler function mapping over w0waCDMCosmology
@jax.jit
def jit_predict_custom(c, z_val, k_out):
    return jaxmapse.get_hmcode_pmm(
        c, z_val,
        k_out=k_out,
        nM=128,  # high-precision mass steps
        linear_pmm_emu=pmm,
        linear_pcb_emu=pcb
    )

# Warmup compilation
print("Compiling JIT pipeline for 100 redshifts and custom k grid...")
t0 = time.time()
k_out_grid, pk_nl_100 = jit_predict_custom(jax_cosmo, z_100, k_custom)
jax.block_until_ready(pk_nl_100)
t1 = time.time()
print(f"Compilation + first run: {(t1 - t0)*1000.0:.2f} ms")

# Benchmark execution
t0 = time.time()
k_out_grid, pk_nl_100 = jit_predict_custom(jax_cosmo, z_100, k_custom)
jax.block_until_ready(pk_nl_100)
t1 = time.time()
print(f"JIT execution for 100 redshifts & 150 custom bins: {(t1 - t0)*1000.0:.2f} ms")
print("Output shapes: k:", k_out_grid.shape, " Pk_nl:", pk_nl_100.shape)


In [ ]:
%timeit jit_predict_custom(jax_cosmo, z_100, k_custom)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.plasma(np.linspace(0.0, 0.8, 5))
indices = [0, 25, 50, 75, 99]
for idx, color in zip(indices, colors):
    ax.loglog(k_out_grid, pk_nl_100[idx], color=color, lw=1.8, label=f'z = {z_100[idx]:.2f}')
ax.set_xlabel('k [Mpc^-1]', fontsize=11)
ax.set_ylabel('P(k) [Mpc^3]', fontsize=11)
ax.set_title('Non-linear Matter Power Spectrum (100-redshift JIT run)', fontsize=12)
ax.grid(True, which='both', alpha=0.25)
ax.legend(fontsize=9)
plt.show()
